# C360 Audience Segmentation (Fresh Demo)
Persona: data scientist; goal: segments + lookalikes entirely in Snowflake.


In [ ]:
import streamlit as st
import altair as alt
import snowflake.snowpark.modin.plugin
from snowflake.snowpark import functions as F
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.window import Window
from snowflake.ml.modeling.preprocessing import StandardScaler
from snowflake.ml.modeling.cluster import KMeans
session = get_active_session()
session.use_database('CROCEVIA_DB'); session.use_schema('GOLD_ANALYTICS')
sales = 'CROCEVIA_DB.BRONZE_DATA.CROCEVIA_SALES_20PCT_STORES'
crm = 'CROCEVIA_DB.RAW_DATA.CROCEVIA_CRM'
products = 'CROCEVIA_DB.BRONZE_DATA.CROCEVIA_PRODUCTS'
st.write('Session ready.')


## Data checks (SnowCLI)
Verified source schemas using connection `SFSENORTHAMERICA-EDDGCPFRANKFURT` before coding. Key findings used to adapt the notebook:
- Sales (`CROCEVIA_DB.BRONZE_DATA.CROCEVIA_SALES_20PCT_STORES`): `ORDER_ID`, `SALE_DATE`, `PRODUCT_ID`, `QUANTITY`, `SALES_PRICE_EURO`, `CUSTOMER_ID` present.
- CRM (`CROCEVIA_DB.RAW_DATA.CROCEVIA_CRM`): `CUSTOMER_ID`, `REGISTRATION_DATE`, `EMAIL`, `CITY`, `MARKETING_OPT_IN` present; we deduplicate to 1 row per `CUSTOMER_ID`.
- Products (`CROCEVIA_DB.BRONZE_DATA.CROCEVIA_PRODUCTS`): product name column is `PRODUCT` (not `PRODUCT_NAME`); categories under `PRODUCT_CATEGORY`/`PRODUCT_SUBCATEGORY`; `PRICE` is stored as text. Dynamic mapping handles these variants.
These checks ensure our dynamic column selection and joins match the live schemas.


## Feature explanations
- Recency (RECENCY_DAYS): days since last purchase: `datediff(day, LAST_PURCHASE_DATE, current_date)`.
- Frequency (PURCHASE_FREQUENCY): distinct order count per customer.
- Monetary (MONETARY_VALUE): sum of `SALES_PRICE_EURO` per customer.
- Units (UNITS): total quantity purchased.
- Product variety (PRODUCT_VARIETY): distinct `PRODUCT_ID`s purchased.

Why scaling: `StandardScaler` normalizes features to comparable scale (0-mean, unit-variance) which KMeans expects.
Why KMeans k=5: fixed for demo clarity; in production, choose k via elbow/silhouette and business interpretability.
Labels: `SEGMENT_LABEL` is derived from the model's prediction column for downstream grouping.
Lookalikes: we compute Euclidean distance in scaled feature space to the top-buyer centroid and rank customers.
Reference: [Snowflake ML APIs](https://docs.snowflake.com/en/developer-guide/snowpark-ml/reference/latest/index-standard).


In [ ]:
# Schema introspection + dynamic selects
sales_df = session.table(sales).select('CUSTOMER_ID','ORDER_ID','SALE_DATE','PRODUCT_ID','QUANTITY','SALES_PRICE_EURO')
crm_tbl = session.table(crm); crm_cols = [f.name for f in crm_tbl.schema.fields]
crm_keep = [c for c in ['CUSTOMER_ID','REGISTRATION_DATE','EMAIL','CITY','MARKETING_OPT_IN'] if c in crm_cols]
crm_df = crm_tbl.select(*crm_keep)
prod_tbl = session.table(products); prod_cols = [f.name for f in prod_tbl.schema.fields]
name_col = 'PRODUCT_NAME' if 'PRODUCT_NAME' in prod_cols else ('PRODUCT' if 'PRODUCT' in prod_cols else None)
prod_keep = ['PRODUCT_ID'] + ([name_col] if name_col else [])
products_df = prod_tbl.select(*prod_keep)
st.write('Sources loaded with dynamic columns.')


In [ ]:
# CRM dedup (latest registration) + join-safety guard
crm_dedup = (crm_df
    .with_column('row_num', F.row_number().over(Window.partition_by('CUSTOMER_ID').order_by(F.col('REGISTRATION_DATE').desc_nulls_last())))
    .filter(F.col('row_num')==1).drop('row_num'))
rfm_base = (sales_df.group_by('CUSTOMER_ID')
    .agg(F.max('SALE_DATE').alias('LAST_PURCHASE_DATE'),
         F.sum('SALES_PRICE_EURO').alias('MONETARY_VALUE'),
         F.count_distinct('ORDER_ID').alias('PURCHASE_FREQUENCY'),
         F.sum('QUANTITY').alias('UNITS'),
         F.count_distinct('PRODUCT_ID').alias('PRODUCT_VARIETY'))
)
rfm_enriched = (rfm_base
    .with_column('RECENCY_DAYS', F.datediff('day', F.col('LAST_PURCHASE_DATE'), F.current_date()))
    .join(crm_dedup, 'CUSTOMER_ID', 'left')
)
total = rfm_enriched.count(); uniq = rfm_enriched.select(F.count_distinct('CUSTOMER_ID')).collect()[0][0]
if total!=uniq:
    st.warning(f'Join produced {total-uniq:,} duplicate rows; enforcing one row per customer.')
    rfm_enriched = rfm_enriched.drop_duplicates(['CUSTOMER_ID'])
st.write('Feature set ready.')


In [ ]:
# Modeling: scale + kmeans + safe writes
feature_cols = ['RECENCY_DAYS','PURCHASE_FREQUENCY','MONETARY_VALUE','UNITS','PRODUCT_VARIETY']
df_for_model = rfm_enriched
for c in feature_cols: df_for_model = df_for_model.with_column(c, F.col(c).cast('DOUBLE'))
scaler = StandardScaler(input_cols=feature_cols, output_cols=[f'SCALED_{c}' for c in feature_cols])
scaled = scaler.fit(df_for_model).transform(df_for_model)
kmeans = KMeans(n_clusters=5, input_cols=[f'SCALED_{c}' for c in feature_cols], random_state=13)
pred = kmeans.fit(scaled).transform(scaled)
if 'SEGMENT_LABEL' not in pred.columns:
    base_pred_col = 'PREDICTION' if 'PREDICTION' in pred.columns else None
    pred = pred.with_column('SEGMENT_LABEL', F.col(base_pred_col)) if base_pred_col else pred
for sc in [f'SCALED_{c}' for c in feature_cols]: pred = pred.with_column(sc, F.coalesce(F.col(sc), F.lit(0.0)))
pred = pred.drop_duplicates(['CUSTOMER_ID'])
pred = pred.with_column('RUN_KEY', F.to_char(F.current_date(), 'YYYYMMDD')).with_column('MODEL_VERSION', F.to_char(F.current_timestamp(), 'YYYYMMDDHH24MISS'))
pred.write.mode('append').save_as_table('CROCEVIA_DB.GOLD_ANALYTICS.C360_CUSTOMER_SEGMENTS_FRESH')
st.success('Segments written to *_FRESH table.')


## Why this matters
Alex (retail media DS) needs actionable shopper segments quickly. This demo shows end-to-end segmentation and lookalikes fully inside Snowflake using Snowpark ML, with governance-friendly, idempotent writes.


In [ ]:
st.markdown('### KPI spotlight')
seg_candidates = []
if 'SEGMENT_LABEL' in pred.columns: seg_candidates.append('SEGMENT_LABEL')
if 'PREDICTION' in pred.columns: seg_candidates.append('PREDICTION')
for _c in pred.columns:
    _u=_c.upper()
    if ('PREDICT' in _u or 'CLUSTER' in _u) and _c not in seg_candidates:
        seg_candidates.append(_c)
seg_col = None
for _c in seg_candidates:
    if pred.filter(F.col(_c).is_not_null()).limit(1).count()>0:
        seg_col=_c; break
tot_customers = pred.select(F.count_distinct('CUSTOMER_ID')).collect()[0][0]
total_revenue = float((pred.select(F.sum('MONETARY_VALUE')).collect()[0][0] or 0)) if 'MONETARY_VALUE' in pred.columns else 0.0
active_days = rfm_enriched.select(F.count_distinct('LAST_PURCHASE_DATE')).collect()[0][0]
num_segments = pred.filter(F.col(seg_col).is_not_null()).select(F.count_distinct(seg_col)).collect()[0][0] if seg_col else 0
top_seg_id='N/A'; top_seg_pct=0.0
if seg_col and num_segments>0:
    _row=(pred.filter(F.col(seg_col).is_not_null()).group_by(seg_col).agg(F.count_distinct('CUSTOMER_ID').alias('CNT')).sort(F.col('CNT').desc()).limit(1).collect()[0])
    top_seg_id=_row[0]; top_seg_cnt=_row[1]; top_seg_pct=round(100.0*top_seg_cnt/max(tot_customers,1),1)
c1,c2,c3,c4 = st.columns(4)
c1.metric('Customers', f'{tot_customers:,}')
c2.metric('Revenue (€)', f'{total_revenue:,.0f}')
c3.metric('Active days', f'{active_days:,}')
c4.metric('Segments / Top segment', f'{num_segments} / {top_seg_pct}%')
st.caption(f'Top buyer segment = {top_seg_id}')


In [ ]:
st.markdown('### Audience intelligence')
left,right = st.columns(2)
with left:
    st.subheader('Segment distribution')
    if seg_col:
        seg_counts=(pred.filter(F.col(seg_col).is_not_null()).group_by(seg_col).agg(F.count_distinct('CUSTOMER_ID').alias('CUSTOMERS')).to_pandas())
        chart=alt.Chart(seg_counts).mark_bar(color='#1f77b4').encode(x=alt.X(f'{seg_col}:N', title='Segment'), y=alt.Y('CUSTOMERS:Q', title='Customers'), tooltip=[seg_col,'CUSTOMERS']).properties(width=420, height=240)
        st.altair_chart(chart, use_container_width=True)
    else:
        st.info('No segment labels available.')
with right:
    st.subheader('Recency vs Frequency (sample)')
    cols_exist = all(c in pred.columns for c in ['RECENCY_DAYS','PURCHASE_FREQUENCY'])
    if cols_exist:
        rf_pd = pred.select('RECENCY_DAYS','PURCHASE_FREQUENCY').limit(5000).to_pandas()
        sc=alt.Chart(rf_pd).mark_circle(size=26, opacity=0.35).encode(x=alt.X('RECENCY_DAYS:Q', title='Recency (days)'), y=alt.Y('PURCHASE_FREQUENCY:Q', title='Frequency'), tooltip=['RECENCY_DAYS','PURCHASE_FREQUENCY']).properties(width=420, height=300)
        st.altair_chart(sc, use_container_width=True)
    else:
        st.info('Features not available for scatter.')


In [ ]:
st.markdown('### Segment product recommendations')
if seg_col:
    base = pred.select('CUSTOMER_ID', F.col(seg_col).alias('SEGMENT_LABEL')).join(sales_df, 'CUSTOMER_ID')
    prod_cols = [f.name for f in products_df.schema.fields]
    name_col = 'PRODUCT_NAME' if 'PRODUCT_NAME' in prod_cols else ('PRODUCT' if 'PRODUCT' in prod_cols else None)
    if name_col:
        joined = base.join(products_df.select('PRODUCT_ID', name_col), 'PRODUCT_ID')
        group_cols = ['SEGMENT_LABEL','PRODUCT_ID', name_col]
    else:
        joined = base
        group_cols = ['SEGMENT_LABEL','PRODUCT_ID']
    segment_recs = (joined.group_by(*group_cols).agg(F.sum('SALES_PRICE_EURO').alias('REVENUE')).with_column('PRODUCT_RANK', F.rank().over(Window.partition_by('SEGMENT_LABEL').order_by(F.col('REVENUE').desc()))).filter(F.col('PRODUCT_RANK')<=5))
    st.table(segment_recs.limit(20))
else:
    st.info('No segment labels available; skipping product recs.')


In [ ]:
st.markdown('### Lookalike audience (top N)')
if seg_col:
    scaled_cols = [c for c in pred.columns if c.startswith('SCALED_')]
    if scaled_cols:
        top_seg = pred.filter(F.col(seg_col).is_not_null()).group_by(F.col(seg_col)).agg(F.sum('MONETARY_VALUE').alias('REV') if 'MONETARY_VALUE' in pred.columns else F.count_distinct('CUSTOMER_ID').alias('CNT')).sort(F.col('REV').desc() if 'MONETARY_VALUE' in pred.columns else F.col('CNT').desc()).limit(1).collect()[0][0]
        fv = pred.select('CUSTOMER_ID', F.col(seg_col).alias('SEGMENT_LABEL'), *scaled_cols)
        centroid_row = fv.filter(F.col('SEGMENT_LABEL')==F.lit(top_seg)).agg(*[F.avg(c).alias(f'CENTROID_{c}') for c in scaled_cols]).collect()[0]
        centroid = {c: centroid_row[f'CENTROID_{c}'] for c in scaled_cols}
        dist=None
        for c in scaled_cols:
            d = F.col(c) - F.lit(centroid[c])
            term = d*d
            dist = term if dist is None else dist + term
        look = fv.with_column('DISTANCE_TO_TOP', F.sqrt(dist)).with_column('LOOKALIKE_RANK', F.dense_rank().over(Window.order_by(F.col('DISTANCE_TO_TOP'))))
        st.table(look.select('CUSTOMER_ID','SEGMENT_LABEL','DISTANCE_TO_TOP','LOOKALIKE_RANK').limit(20))
    else:
        st.info('No scaled features; skipping lookalikes.')
else:
    st.info('No segment labels available; skipping lookalikes.')
